# LLM Benchmark — GGUF Runner (llama.cpp)

## Kullanım
1. **A bölümü**: Bir kere çalıştır
2. **B1**: Model ve parametreleri ayarla
3. **B2**: Model indir + server başlat
4. **B3**: Benchmark çalıştır
5. **B4-B6**: Sonuçları gör ve indir
6. **Sonraki model**: B1'e dön, modeli değiştir, B2'den devam et

---
# A) İlk Kurulum (bir kere)

In [ ]:
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    !huggingface-cli login --token {hf_token} --add-to-git-credential
    print('✅ HuggingFace login başarılı')
except Exception as e:
    print(f'⚠️ HF_TOKEN bulunamadı: {e}')

!CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python[server] --force-reinstall --no-cache-dir 2>&1 | tail -5
!pip -q install httpx pydantic hypothesis nest_asyncio huggingface_hub 2>&1 | tail -3
!nvidia-smi | head -12

In [ ]:
import os, sys

ROOT = '/content/llm'
LOG_DIR = f'{ROOT}/logs'
CACHE_DIR = f'{ROOT}/cache'
MODEL_DIR = f'{ROOT}/models'
PROJECT_DIR = f'{ROOT}/benchmark'

for p in [ROOT, LOG_DIR, CACHE_DIR, MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

os.environ['HF_HOME'] = CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE_DIR

REPO_URL = 'https://github.com/orhan-kaplan/benchmark.git'
if os.path.exists(PROJECT_DIR):
    print('📦 Güncelleniyor...')
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/main
else:
    print('📦 Klonlanıyor...')
    !git clone {REPO_URL} {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'✅ Proje hazır: {PROJECT_DIR}')

---
# B) Model Test Döngüsü

In [ ]:
import json

# ╔══════════════════════════════════════════════════════════════════╗
# ║  CONFIG — değiştirmek istediğini yaz, "default" = json'dan al  ║
# ╚══════════════════════════════════════════════════════════════════╝
MODEL           = 1              # ID (sayı) veya isim (string)
TEST_SETS       = ['hard5']      # default = json'dan
MAX_TOKENS      = 'default'      # default = n_ctx - 2048
TEMPERATURE     = 'default'      # default = 0.7
TOP_P           = 'default'      # default = 1.0
N_GPU_LAYERS    = 'default'      # default = json'dan (-1 = tümü GPU)
N_CTX           = 'default'      # default = json'dan (65536)
PORT            = 8090

# ─── Model bul ───
with open(f'{PROJECT_DIR}/benchmark_data/models-llama.json') as f:
    _models = json.load(f)['models']

_m = None
if isinstance(MODEL, int):
    _m = next((x for x in _models if x.get('id') == MODEL), None)
else:
    _m = next((x for x in _models if x['name'] == MODEL), None)

if _m is None:
    for x in _models: print(f"  {x.get('id','?'):2}. {x['name']}")
    raise ValueError(f'Model bulunamadı: {MODEL}')

# ─── Resolve ───
_l = _m.get('llama', {})
_b = _m.get('benchmark', {})
R = lambda val, fallback: fallback if val == 'default' else val

HF_REPO        = _m['hf_repo']
GGUF_FILE      = _m['gguf_file']
MODEL_LABEL    = _m['name']
N_GPU_LAYERS   = R(N_GPU_LAYERS, _l.get('n_gpu_layers', -1))
N_CTX          = R(N_CTX,        _l.get('n_ctx', 65536))
TEMPERATURE    = R(TEMPERATURE,  _b.get('temperature', 0.7))
TOP_P          = R(TOP_P,        _b.get('top_p', 1.0))
TEST_SETS      = R(TEST_SETS,    _b.get('test_sets', ['coding']))
MAX_TOKENS     = R(MAX_TOKENS,   _b.get('max_tokens', N_CTX - 2048))
if MAX_TOKENS > N_CTX - 2048:
    MAX_TOKENS = N_CTX - 2048

VLLM_BASE_URL  = f'http://localhost:{PORT}'
MODEL_PATH     = f'{MODEL_DIR}/{GGUF_FILE}'

print(f'╔{"═"*58}╗')
print(f'║ [{_m.get("id")}] {MODEL_LABEL:50s}   ║')
print(f'╠{"═"*58}╣')
print(f'║  repo:       {HF_REPO[:44]:44s} ║')
print(f'║  file:       {GGUF_FILE[:44]:44s} ║')
print(f'║  n_gpu:      {N_GPU_LAYERS:<44} ║')
print(f'║  n_ctx:      {N_CTX:<44} ║')
print(f'╠{"═"*58}╣')
print(f'║  test_sets:  {str(TEST_SETS)[:44]:44s} ║')
print(f'║  max_tokens: {MAX_TOKENS:<44} ║')
print(f'║  temperature:{TEMPERATURE:<44} ║')
print(f'║  top_p:      {TOP_P:<44} ║')
print(f'╚{"═"*58}╝')

In [ ]:
import subprocess, time, requests

print('🛑 Önceki server durduruluyor...')
subprocess.run('pkill -f llama_cpp.server || true', shell=True, check=False)
time.sleep(2)

# Model indir
if not os.path.exists(MODEL_PATH):
    print(f'📥 İndiriliyor: {GGUF_FILE}')
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id=HF_REPO, filename=GGUF_FILE, local_dir=MODEL_DIR)
    print('✅ İndirildi')
else:
    print(f'✅ Mevcut: {MODEL_PATH}')

# Server başlat
LOG_PATH = f'{LOG_DIR}/llama-{MODEL_LABEL}.log'
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', MODEL_PATH,
    '--n_gpu_layers', str(N_GPU_LAYERS),
    '--n_ctx', str(N_CTX),
    '--host', '0.0.0.0',
    '--port', str(PORT),
]

with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

print(f'🚀 Server başlatıldı (PID: {proc.pid})')
print(f'   n_ctx: {N_CTX}, n_gpu_layers: {N_GPU_LAYERS}')

t0 = time.time()
ready = False
while time.time() - t0 < 600:
    try:
        if requests.get(f'{VLLM_BASE_URL}/v1/models', timeout=3).status_code == 200:
            ready = True; break
    except: pass
    if proc.poll() is not None:
        print(f'❌ Çöktü! (exit: {proc.returncode})')
        with open(LOG_PATH) as f: print(f.read()[-1000:])
        break
    print(f'[{int(time.time()-t0):3d}s] ⏳ Başlatılıyor')
    time.sleep(5)

if ready:
    print(f'\n✅ Server hazır! ({int(time.time()-t0)}s)')
    r = requests.post(f'{VLLM_BASE_URL}/v1/chat/completions',
        json={'model': MODEL_PATH, 'messages': [{'role':'user','content':'Say hello'}], 'max_tokens': 32}, timeout=120)
    if r.status_code == 200:
        msg = r.json()['choices'][0]['message']
        txt = msg.get('content') or 'No content'
        print(f'   💬 {txt[:150]}')
else:
    print(f'\n❌ Timeout!')

In [ ]:
import asyncio, importlib
from pathlib import Path

import benchmark.runner, benchmark.catalog, benchmark.test_sets
import benchmark.api_client, benchmark.metrics, benchmark.storage, benchmark.models
for mod in [benchmark.models, benchmark.storage, benchmark.metrics,
            benchmark.api_client, benchmark.catalog, benchmark.test_sets, benchmark.runner]:
    importlib.reload(mod)

from benchmark.runner import BenchmarkRunner
from benchmark.catalog import ModelCatalog
from benchmark.test_sets import TestSetManager
from benchmark.api_client import APIClient
from benchmark.metrics import MetricsCollector, VRAMTracker
from benchmark.storage import StorageManager
from benchmark.models import BenchmarkRunConfig, GenerationParams, ModelEntry, Backend

data_path = Path(PROJECT_DIR) / 'benchmark_data'
storage = StorageManager(data_path)
catalog = ModelCatalog(data_path)
test_mgr = TestSetManager(data_path)
api_client = APIClient(timeout=600.0, max_retries=3)
metrics_collector = MetricsCollector()
vram_tracker = VRAMTracker()

if catalog.get(MODEL_LABEL) is None:
    catalog.register(ModelEntry(name=MODEL_LABEL, repo=MODEL_PATH, format='GGUF',
                               backend=Backend.LLAMA_CPP, tags=['gguf'], api_endpoint=VLLM_BASE_URL))
else:
    catalog.update(MODEL_LABEL, {'api_endpoint': VLLM_BASE_URL, 'repo': MODEL_PATH})
print(f'Model: {MODEL_LABEL} → {VLLM_BASE_URL}')
print(f'max_tokens={MAX_TOKENS}, temperature={TEMPERATURE}, top_p={TOP_P}')
print(f'test_sets={TEST_SETS}')

runner = BenchmarkRunner(catalog=catalog, test_set_manager=test_mgr, api_client=api_client,
                         metrics_collector=metrics_collector, storage=storage, vram_tracker=vram_tracker)

async def run_benchmarks():
    ids = []
    for ts_name in TEST_SETS:
        print(f"\n{'='*60}")
        print(f'Benchmark: {ts_name} × {MODEL_LABEL}')
        print(f"{'='*60}")
        config = BenchmarkRunConfig(test_set_name=ts_name, model_names=[MODEL_LABEL],
            params=GenerationParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS, top_p=TOP_P))
        result = await runner.run(config)
        ids.append(result.run_id)
        ok = sum(1 for r in result.results if r.success)
        fail = sum(1 for r in result.results if not r.success)
        print(f'✅ {result.run_id} | {ok} başarılı, {fail} başarısız')
    await api_client.close()
    return ids

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    run_ids = asyncio.run(run_benchmarks())
except RuntimeError:
    run_ids = asyncio.run(run_benchmarks())

print(f"\n🏁 Tamamlandı: {run_ids}")

In [ ]:
from benchmark.reporter import ReportGenerator
reporter = ReportGenerator(storage=storage)

for run_id in run_ids:
    print(f"\n{'='*60}")
    print(f'Rapor: {run_id}')
    print(f"{'='*60}")
    summary = reporter.generate_summary(run_id)
    print(f'  ✅ {summary.successful_results} | ❌ {summary.failed_results}')
    for r in storage.read_results(run_id):
        s = '✅' if r.get('success') else '❌'
        m = r.get('metrics') or {}
        tok = m.get('completion_tokens') or 0
        tps = m.get('tokens_per_second') or 0
        t = (m.get('total_time_ms') or 0) / 1000
        print(f'  {s} {r["prompt_id"]:15s} | {tok:5d} tok | {tps:6.1f} t/s | {t:5.1f}s')
        if r.get('error'): print(f'     ⚠️ {r["error"][:100]}')
    reporter.export_json(run_id, storage.runs_dir / run_id / 'report.json')
    reporter.export_csv(run_id, storage.runs_dir / run_id / 'report.csv')

In [ ]:
INSPECT_PROMPT = None  # None=hepsi, veya 'h5-coding'

for run_id in run_ids:
    for r in storage.read_results(run_id):
        if INSPECT_PROMPT and r['prompt_id'] != INSPECT_PROMPT: continue
        print(f"\n{'─'*60}")
        print(f"{r['prompt_id']} | {r['model_name']}")
        print(f"{'─'*60}")
        print(r.get('response_text') or r.get('error') or 'Yanıt yok')

In [ ]:
import shutil, json as _json
from google.colab import files

for run_id in run_ids:
    run_dir = str(storage.runs_dir / run_id)
    with open(f'{run_dir}/meta.json') as f:
        ts = _json.load(f).get('config',{}).get('test_set_name','unknown')
    name = f'{MODEL_LABEL}_{ts}_{run_id}'
    shutil.make_archive(f'/content/{name}', 'zip', run_dir)
    print(f'📦 {name}.zip')
    files.download(f'/content/{name}.zip')

---
# C) Opsiyonel — ngrok + keepalive

In [ ]:
try:
    from google.colab import userdata
    from pyngrok import ngrok
    ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
    ngrok.kill()
    tunnel = ngrok.connect(PORT, 'http')
    print(f'🌐 {tunnel.public_url}')
    print(f'API → {tunnel.public_url}/v1')
except Exception as e:
    print(f'ngrok başarısız: {e}')

In [ ]:
import time, requests
print('Canlı tutma aktif. Durdurmak için interrupt et.')
while True:
    try: s = '✅' if requests.get(f'{VLLM_BASE_URL}/v1/models', timeout=5).status_code == 200 else '⚠️'
    except: s = '❌'
    print(f'{s} {time.strftime("%H:%M:%S")}')
    time.sleep(30)